# Análisis de artefactos PKL - Random Forest

Este notebook permite cargar los archivos `.pkl` del modelo Random Forest, inspeccionar su contenido y generar métricas y gráficos analíticos.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    roc_curve
)

from Processing_RF import load_and_preprocess_data

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 100)


In [ ]:
MODEL_DIR = Path.cwd()

# Si se ejecuta desde otra carpeta, ajustar automáticamente a Models/Random_Forest
if MODEL_DIR.name != "Random_Forest":
    candidate = MODEL_DIR / "Models" / "Random_Forest"
    if candidate.exists():
        MODEL_DIR = candidate

PKL_FILES = {
    "modelo_base": MODEL_DIR / "random_forest_model.pkl",
    "modelo_optimizado": MODEL_DIR / "random_forest_optimizado.pkl",
    "scaler": MODEL_DIR / "scaler.pkl",
    "feature_names": MODEL_DIR / "feature_names.pkl",
}

for name, path in PKL_FILES.items():
    print(f"{name:18} -> {path} | existe={path.exists()}")


In [ ]:
artefactos = {}
for name, path in PKL_FILES.items():
    if path.exists():
        artefactos[name] = joblib.load(path)

resumen = []
for name, obj in artefactos.items():
    fila = {
        "artefacto": name,
        "tipo": type(obj).__name__,
    }

    if hasattr(obj, "__len__") and not isinstance(obj, (str, bytes)):
        try:
            fila["longitud"] = len(obj)
        except TypeError:
            fila["longitud"] = np.nan
    else:
        fila["longitud"] = np.nan

    if hasattr(obj, "n_features_in_"):
        fila["n_features_in_"] = int(obj.n_features_in_)

    resumen.append(fila)

pd.DataFrame(resumen)


In [ ]:
if "feature_names" in artefactos:
    feature_names = artefactos["feature_names"]
    print(f"Total de features: {len(feature_names)}")
    pd.DataFrame({"feature": feature_names}).head(15)
else:
    print("No se encontró feature_names.pkl")


In [ ]:
X_train, X_test, y_train, y_test, _, _ = load_and_preprocess_data()

print(f"X_train: {X_train.shape} | X_test: {X_test.shape}")
print(f"Distribución y_test:
{pd.Series(y_test).value_counts(normalize=True).rename('proporcion')}")


In [ ]:
def evaluar_modelo(nombre, modelo, X, y):
    y_pred = modelo.predict(X)
    y_prob = modelo.predict_proba(X)[:, 1]

    return {
        "modelo": nombre,
        "accuracy": accuracy_score(y, y_pred),
        "precision": precision_score(y, y_pred, zero_division=0),
        "recall": recall_score(y, y_pred, zero_division=0),
        "f1": f1_score(y, y_pred, zero_division=0),
        "auc_roc": roc_auc_score(y, y_prob),
        "y_pred": y_pred,
        "y_prob": y_prob,
    }

resultados = []
for clave in ["modelo_base", "modelo_optimizado"]:
    modelo = artefactos.get(clave)
    if modelo is not None:
        resultados.append(evaluar_modelo(clave, modelo, X_test, y_test))

metricas_df = pd.DataFrame([{k: v for k, v in r.items() if not k.startswith("y_")} for r in resultados])
metricas_df.sort_values("recall", ascending=False) if not metricas_df.empty else metricas_df


In [ ]:
if resultados:
    fig, axes = plt.subplots(1, len(resultados), figsize=(7 * len(resultados), 5))
    if len(resultados) == 1:
        axes = [axes]

    for ax, r in zip(axes, resultados):
        cm = confusion_matrix(y_test, r["y_pred"])
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax)
        ax.set_title(f"Matriz de confusión - {r['modelo']}")
        ax.set_xlabel("Predicción")
        ax.set_ylabel("Real")

    plt.tight_layout()
    plt.show()
else:
    print("No hay modelos cargados para graficar.")


In [ ]:
if resultados:
    plt.figure(figsize=(8, 6))
    for r in resultados:
        fpr, tpr, _ = roc_curve(y_test, r["y_prob"])
        plt.plot(fpr, tpr, label=f"{r['modelo']} (AUC={r['auc_roc']:.3f})")

    plt.plot([0, 1], [0, 1], "k--", alpha=0.7)
    plt.title("Curva ROC comparativa")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.legend()
    plt.show()


In [ ]:
for clave in ["modelo_base", "modelo_optimizado"]:
    modelo = artefactos.get(clave)
    if modelo is None or not hasattr(modelo, "feature_importances_"):
        continue

    names = artefactos.get("feature_names", [f"f_{i}" for i in range(len(modelo.feature_importances_))])
    importancias = pd.DataFrame({
        "feature": names,
        "importance": modelo.feature_importances_
    }).sort_values("importance", ascending=False).head(15)

    plt.figure(figsize=(10, 6))
    sns.barplot(data=importancias, x="importance", y="feature", palette="viridis")
    plt.title(f"Top 15 importancias - {clave}")
    plt.xlabel("Importancia")
    plt.ylabel("Feature")
    plt.tight_layout()
    plt.show()
